# 教学模块三：训练、评估与可视化
# Teaching Module 3: Training, Evaluation, and Visualization

欢迎来到项目的最后一个核心模块！

Welcome to the final core module of the project!

至此，我们已经拥有了：

At this point, we already have:

1.  **高质量的数据**：通过 `EEGDataset` 类，我们可以方便地获取经过滤波、标准化和窗口化的EEG数据样本。

1  **High-quality Data**: Through the `EEGDataset` class, we can easily access filtered, standardized, and windowed EEG data samples.

2.  **强大的模型**：我们构建了一个 `EEGBiFormer` 模型，它结合了CNN和Transformer的优点。

2  **A Powerful Model**: We have built an `EEGBiFormer` model that combines the advantages of CNNs and Transformers.

现在，是时候将这两者结合起来，通过**模型训练**，让模型从数据中学习知识了。

Now, it's time to bring them together and let the model learn from the data through **model training**.

### 本模块学习目标：

### Learning Objectives for This Module:

1.  **理解训练循环**：掌握PyTorch中模型训练的核心四步：前向传播、计算损失、反向传播、更新参数。

1  **Understand the Training Loop**: Master the four core steps of model training in PyTorch: forward pass, loss calculation, backward pass, and parameter update.

2  **学习高级训练技巧**：

2.  **Learn Advanced Training Techniques**:

    * 如何处理**类别不平衡**问题。

    * How to handle the **class imbalance** problem.

    * 如何使用**学习率调度器**和**早停机制**来防止过拟合。

    * How to use a **learning rate scheduler** and **early stopping** to prevent overfitting.

    * 如何通过**梯度裁剪**来增强训练稳定性。

    * How to enhance training stability with **gradient clipping**.

3.  **掌握模型评估与可视化**：学习如何绘制损失/准确率曲线和混淆矩阵，并从这些图表中分析模型性能。

3  **Master Model Evaluation and Visualization**: Learn how to plot loss/accuracy curves and confusion matrices, and analyze model performance from these charts.

4.  **整合项目**：将所有模块串联起来，运行一个完整的端到端训练流程。

4 **Integrate the Project**: Connect all modules to run a complete end-to-end training pipeline.

## 原理剖析（一）：核心训练循环

## Principle Analysis (Part 1): The Core Training Loop

在PyTorch中，训练一个神经网络就像一个不断重复的循环，每一轮（Epoch）都会遍历整个训练数据集，并对模型进行微调。

In PyTorch, training a neural network is like a repetitive loop; each epoch iterates through the entire training dataset to fine-tune the model.

在每一批次（Batch）数据的处理中，都包含以下四个核心步骤：

The processing of each batch of data involves the following four core steps:

1.  **前向传播 (Forward Pass)**

    `outputs = model(data)`

    我们将一批数据输入模型，模型根据其当前的内部参数（权重）进行计算，得出一个预测结果。

    We feed a batch of data into the model, which then computes a prediction based on its current internal parameters (weights).

2.  **计算损失 (Calculate Loss)**

    `loss = criterion(outputs, labels)`

    我们使用一个**损失函数**（如此处使用的**交叉熵损失 `CrossEntropyLoss`**）来比较模型的预测结果和真实的标签。

    We use a **loss function** (like the **CrossEntropyLoss** used here) to compare the model's predictions with the true labels.

    损失值（Loss）是一个数字，它量化了模型“错得有多离谱”。

    The loss is a number that quantifies how "wrong" the model's prediction is.

    损失值越大，说明模型的预测越差。

    The larger the loss value, the worse the model's prediction.

3.  **反向传播 (Backward Pass)**

    `loss.backward()`

    这是最神奇的一步。

    This is the most magical step.

    PyTorch会自动计算损失值相对于模型中每一个参数的**梯度（Gradient）**。

    PyTorch automatically calculates the **gradient** of the loss with respect to every parameter in the model.

    梯度指明了“方向”：我们应该如何调整每个参数，才能让损失值下降得最快。

    The gradient indicates the "direction": how we should adjust each parameter to make the loss decrease the fastest.

4.  **更新参数 (Update Parameters)**

    `optimizer.step()`

    **优化器**（如此处使用的`AdamW`）根据上一步计算出的梯度，对模型的所有参数进行一次微小的更新。

    The **optimizer** (like the `AdamW` used here) performs a small update to all of the model's parameters based on the gradients calculated in the previous step.

通过成千上万次重复这个循环，模型中的参数会逐渐被优化，使得模型在面对训练数据时，计算出的损失值越来越小，预测越来越准确。

By repeating this cycle thousands of times, the model's parameters are gradually optimized, causing the calculated loss on the training data to get smaller and the predictions to become more accurate.

## 原理剖析（二）：高级训练与评估技巧
## Principle Analysis (Part 2): Advanced Training and Evaluation Techniques

仅仅有核心循环是不够的，为了让训练更高效、稳定，结果更可靠，我们引入了以下几种关键技术：

A core loop alone is not enough. To make training more efficient, stable, and the results more reliable, we introduce the following key techniques:

#### A. 处理类别不平衡
#### A. Handling Class Imbalance
* **问题**：我们的训练数据中，“专注”类的样本远多于“放松”类。
* **Problem**: In our training data, samples of the "focus" class far outnumber those of the "rest" class.
* **如果不做处理**，模型可能会“偷懒”，倾向于一直预测样本多的类别，这样也能获得不错的表面准确率。
* **If left unhandled**, the model might get "lazy" and tend to always predict the majority class, which can still achieve a deceptively high accuracy.
* **解决方案**：**类别加权**。
* **Solution**: **Class Weighting**.
* **我们在计算损失时**，给样本量少的类别（放松）分配一个更高的权重。
* **When calculating the loss**, we assign a higher weight to the minority class ("rest").
* **这样一来**，如果模型预测错了“放松”类的样本，会产生比预测错“专注”类更大的损失。
* **This way**, misclassifying a "rest" sample incurs a larger penalty than misclassifying a "focus" sample.
* **这迫使模型**必须认真学习少数类的特征。
* **This forces the model** to learn the features of the minority class more carefully.

#### B. 优化器与学习率调度器
#### B. Optimizer and Learning Rate Scheduler
* **优化器 `AdamW`**：是一个非常强大且流行的优化器，它能为模型的不同参数自动适应学习率，并有更好的权重衰减处理，通常能让模型收敛得更快更好。
* **Optimizer `AdamW`**: A very powerful and popular optimizer that adapts the learning rate for different parameters of the model and handles weight decay more effectively, often leading to faster and better convergence.
* **学习率调度器 `ReduceLROnPlateau`**：学习率决定了我们每次更新参数的“步子”迈多大。
* **Learning Rate Scheduler `ReduceLROnPlateau`**: The learning rate determines the "step size" of each parameter update.
* **一个固定的学习率**很难做到完美。
* **A fixed learning rate** is rarely perfect.
* **这个调度器**会监控**验证集**的损失，如果发现损失在几轮内都不再下降（即模型遇到了瓶颈），它就会自动**降低学习率**。
* **This scheduler** monitors the loss on the **validation set**. If the loss stops decreasing for several epochs (i.e., the model hits a plateau), it will automatically **reduce the learning rate**.
* **这就像**在接近目标时放慢脚步，以便更精确地找到最优解。
* **This is like** slowing down as you approach a target to find the optimal solution more precisely.

#### C. 防止过拟合与早停机制
#### C. Preventing Overfitting and Early Stopping
* **过拟合**：指模型在训练数据上表现完美，但在新的、未见过的数据（如验证集）上表现很差。
* **Overfitting**: Occurs when a model performs perfectly on training data but poorly on new, unseen data (like the validation set).
* **这是因为它**“死记硬背”了训练数据的特征，而没有学到通用的规律。
* **This happens because** it has "memorized" the features of the training data instead of learning generalizable patterns.
* **早停 (Early Stopping)**：我们持续监控模型在验证集上的准确率。
* **Early Stopping**: We continuously monitor the model's accuracy on the validation set.
* **如果准确率**连续很多轮（例如15轮）都没有新的提升，我们就认为模型已经达到了它的最佳状态，并提前终止训练。
* **If the accuracy** does not improve for many consecutive epochs (e.g., 15), we assume the model has reached its peak performance and stop the training early.
* **同时**，我们只保存那个在验证集上准确率最高的模型版本。
* **At the same time**, we only save the model version that achieved the highest accuracy on the validation set.

#### D. 评估与可视化
#### D. Evaluation and Visualization
* **损失/准确率曲线**：通过绘制训练过程中的性能曲线，我们可以清晰地看到模型的学习进程。
* **Loss/Accuracy Curves**: By plotting performance curves during training, we can clearly see the model's learning progress.
* **如果训练准确率**持续上升，而验证准确率停滞或下降，这就是一个典型的过拟合信号。
* **If the training accuracy** continues to rise while the validation accuracy stagnates or falls, it's a classic sign of overfitting.
* **混淆矩阵**：它能清晰地展示出模型将每个类别都预测成了什么。
* **Confusion Matrix**: It clearly shows what each class was predicted as by the model.
* **通过混淆矩阵**，我们可以分析模型的“薄弱环节”，例如，它是不是更容易把“放松”误判为“专注”。
* **Through the confusion matrix**, we can analyze the model's "weak spots," for instance, whether it's more prone to misclassifying "rest" as "focus."

## 代码设计1：评估与可视化函数
## Code Design 1: Evaluation and Visualization Functions

只看训练过程中的数字是不够的。

Looking at just the numbers during training is not enough.

我们需要将结果可视化，以便更直观地理解模型的性能。

We need to visualize the results to understand the model's performance more intuitively.

我们定义两个函数：

We will define two functions:

1  `plot_curves`：绘制训练和验证过程中的损失（Loss）和准确率（Accuracy）曲线。

1.  `plot_curves`: To plot the loss and accuracy curves for both the training and validation phases.

2  通过这些曲线，我们可以判断模型是否正常收敛，以及是否出现了过拟合。

2.  From these curves, we can determine if the model is converging properly and whether overfitting has occurred.

3  `plot_confusion_matrix`：在训练结束后，用最佳模型在验证集上进行预测，并生成一个混淆矩阵。

3.  `plot_confusion_matrix`: After training is complete, it uses the best model to make predictions on the validation set and generates a confusion matrix.

4  混淆矩阵可以告诉我们模型将每个类别都预测成了什么，帮助我们分析模型的“薄弱环节”。

4.  The confusion matrix tells us what each class was predicted as, helping us analyze the model's "weak spots."

In [1]:
# 导入可视化和评估相关的库
# Import libraries related to visualization and evaluation
import matplotlib.pyplot as plt
# 从 scikit-learn 中导入用于计算和显示混淆矩阵的工具
# Import tools from scikit-learn for calculating and displaying the confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
# 从 tqdm 库导入 tqdm，用于在循环中显示一个美观的进度条
# Import tqdm from the tqdm library to display a nice progress bar in loops
from tqdm import tqdm 
# 从 PyTorch 的优化器模块中导入学习率调度器 ReduceLROnPlateau
# Import the learning rate scheduler ReduceLROnPlateau from PyTorch's optimizer module
from torch.optim.lr_scheduler import ReduceLROnPlateau

# 定义一个函数，用于绘制训练过程中的损失和准确率曲线
# Define a function to plot the loss and accuracy curves during training
def plot_curves(train_loss_history, val_loss_history, train_acc_history, val_acc_history):
    """
    绘制并保存训练和验证的损失与准确率曲线。
    Plots and saves the training and validation loss and accuracy curves.
    """
    # 获取总的训练轮数
    # Get the total number of training epochs
    num_epochs = len(train_loss_history)
    # 创建一个从1到总轮数的整数序列，用于绘图的x轴
    # Create a sequence of integers from 1 to the total number of epochs for the plot's x-axis
    epochs = range(1, num_epochs + 1)
    
    # 创建一个大小为 15x5 英寸的图形区域
    # Create a figure area with a size of 15x5 inches
    plt.figure(figsize=(15, 5))
    
    # 绘制损失曲线
    # Plot the loss curves
    # 在 1x2 的网格中，选择第一个子图
    # In a 1x2 grid, select the first subplot
    plt.subplot(1, 2, 1)
    # 绘制训练损失曲线，设置标签、颜色和标记样式
    # Plot the training loss curve, setting the label, color, and marker style
    plt.plot(epochs, train_loss_history, label='Training Loss', color='blue', marker='o')
    # 绘制验证损失曲线
    # Plot the validation loss curve
    plt.plot(epochs, val_loss_history, label='Validation Loss', color='red', marker='o')
    # 设置子图的标题
    # Set the title for the subplot
    plt.title('Training and Validation Loss')
    # 设置x轴的标签
    # Set the label for the x-axis
    plt.xlabel('Epoch')
    # 设置y轴的标签
    # Set the label for the y-axis
    plt.ylabel('Loss')
    # 显示图例
    # Display the legend
    plt.legend()
    # 显示网格线
    # Display grid lines
    plt.grid(True)
    
    # 绘制准确率曲线
    # Plot the accuracy curves
    # 在 1x2 的网格中，选择第二个子图
    # In a 1x2 grid, select the second subplot
    plt.subplot(1, 2, 2)
    # 绘制训练准确率曲线
    # Plot the training accuracy curve
    plt.plot(epochs, train_acc_history, label='Training Accuracy', color='green', marker='o')
    # 绘制验证准确率曲线
    # Plot the validation accuracy curve
    plt.plot(epochs, val_acc_history, label='Validation Accuracy', color='orange', marker='o')
    # 设置子图的标题
    # Set the title for the subplot
    plt.title('Training and Validation Accuracy')
    # 设置x轴的标签
    # Set the label for the x-axis
    plt.xlabel('Epoch')
    # 设置y轴的标签
    # Set the label for the y-axis
    plt.ylabel('Accuracy')
    # 显示图例
    # Display the legend
    plt.legend()
    # 显示网格线
    # Display grid lines
    plt.grid(True)
    
    # 自动调整子图参数，使之填充整个图像区域
    # Automatically adjust subplot parameters to give a tight layout
    plt.tight_layout()
    # 将生成的图像保存为 'training_curves.png' 文件
    # Save the generated figure as 'training_curves.png'
    plt.savefig('training_curves.png')
    # 打印一条消息，确认图片已保存
    # Print a message to confirm the plot has been saved
    print("\n成功将训练曲线图保存为 'training_curves.png'")

# 定义一个函数，用于绘制混淆矩阵
# Define a function to plot the confusion matrix
def plot_confusion_matrix(model, val_loader, device, class_names):
    """
    加载最佳模型，在验证集上进行预测，并生成/保存混淆矩阵图。
    Loads the best model, makes predictions on the validation set, and generates/saves the confusion matrix plot.
    """
    # 打印提示信息
    # Print an informational message
    print("\n正在使用最佳模型在验证集上生成混淆矩阵...")
    # 使用 try-except 块来处理可能发生的错误，如文件找不到
    # Use a try-except block to handle potential errors, such as file not found
    try:
        # 从 'best_model.pth' 文件中加载保存的最好模型的状态字典（权重）
        # Load the saved state dictionary (weights) of the best model from the 'best_model.pth' file
        model.load_state_dict(torch.load('best_model.pth'))
        # 将模型移动到指定的设备（CPU或GPU）
        # Move the model to the specified device (CPU or GPU)
        model.to(device)
        # 将模型设置为评估模式
        # Set the model to evaluation mode
        model.eval()

        # 初始化两个空列表，用于存储所有的真实标签和预测标签
        # Initialize two empty lists to store all true labels and predicted labels
        all_labels, all_preds = [], []

        # 使用 torch.no_grad() 上下文管理器，在该块内不计算梯度
        # Use the torch.no_grad() context manager to disable gradient calculation within this block
        with torch.no_grad():
            # 遍历验证集数据加载器中的所有批次
            # Iterate over all batches in the validation data loader
            for data, labels in val_loader:
                # 将数据和标签移动到指定设备，并转换数据类型
                # Move data and labels to the specified device and convert their data types
                data, labels_tensor = data.to(device).float(), labels.to(device).long()
                # 模型进行前向传播，得到输出
                # Perform a forward pass with the model to get the output
                outputs = model(data)
                # 获取输出中概率最大的类别的索引作为预测结果
                # Get the index of the maximum value in the output as the prediction
                preds = outputs.argmax(1)
                
                # 将当前批次的真实标签（转换回CPU上的numpy数组）添加到列表中
                # Extend the list of all true labels with the current batch's labels (converted back to a numpy array on the CPU)
                all_labels.extend(labels_tensor.cpu().numpy())
                # 将当前批次的预测结果添加到列表中
                # Extend the list of all predictions with the current batch's predictions
                all_preds.extend(preds.cpu().numpy())
        
        # 使用 sklearn 的 ConfusionMatrixDisplay 工具，根据真实标签和预测结果生成混淆矩阵图
        # Use scikit-learn's ConfusionMatrixDisplay tool to generate a confusion matrix plot from the true and predicted labels
        ConfusionMatrixDisplay.from_predictions(
            y_true=all_labels, y_pred=all_preds, # 传入真实标签和预测标签
            # Pass in the true labels and predicted labels
            display_labels=class_names, cmap=plt.cm.Blues, xticks_rotation='horizontal' # 设置显示的类别名、颜色主题和x轴刻度标签的旋转角度
            # Set the display labels, color map, and x-axis tick label rotation
        )
        
        # 设置混淆矩阵图的标题
        # Set the title for the confusion matrix plot
        plt.title('Confusion Matrix for Validation Set')
        # 自动调整布局
        # Adjust the layout to be tight
        plt.tight_layout()
        # 将图像保存为 'confusion_matrix.png' 文件
        # Save the plot as 'confusion_matrix.png'
        plt.savefig('confusion_matrix.png')
        # 打印保存成功的消息
        # Print a success message
        print("成功将混淆矩阵图保存为 'confusion_matrix.png'")
    
    # 如果 'best_model.pth' 文件找不到，则捕获这个特定的错误
    # If the 'best_model.pth' file is not found, catch this specific error
    except FileNotFoundError:
        # 打印提示信息，并跳过生成过程
        # Print a message and skip the generation process
        print("找不到 'best_model.pth'。跳过混淆矩阵生成。")
    # 捕获其他所有可能的异常
    # Catch all other possible exceptions
    except Exception as e:
        # 打印详细的错误信息
        # Print the detailed error message
        print(f"生成混淆矩阵时出错: {e}")

# 打印一条消息，确认函数已成功定义
# Print a message to confirm the functions have been successfully defined
print("评估与可视化函数 'plot_curves' 和 'plot_confusion_matrix' 定义成功！")

评估与可视化函数 'plot_curves' 和 'plot_confusion_matrix' 定义成功！


## 代码设计2：核心训练逻辑 - train_model 函数
## Code Design 2: Core Training Logic - The `train_model` Function

这是整个项目的“引擎室”。

This is the "engine room" of the entire project.

`train_model` 函数封装了模型训练和评估的完整循环。

The `train_model` function encapsulates the complete cycle of model training and evaluation.

它包含了许多现代深度学习训练的最佳实践：

It incorporates many best practices of modern deep learning training:

* **处理类别不平衡**：通过计算类别权重，让模型在训练时更加关注样本量少的类别。
* **Handling Class Imbalance**: By calculating class weights, it makes the model pay more attention to the minority class during training.
* **学习率调度**：根据验证集性能动态调整学习率，帮助模型更好地收敛。
* **Learning Rate Scheduling**: Dynamically adjusts the learning rate based on validation set performance to help the model converge better.
* **早停机制**：在模型性能不再提升时及时停止训练，防止过拟合。
* **Early Stopping**: Stops training promptly when model performance no longer improves, preventing overfitting.
* **梯度裁剪**：防止梯度爆炸，增强训练稳定性。
* **Gradient Clipping**: Prevents exploding gradients to enhance training stability.
* **最佳模型保存**：只保留验证集上表现最好的模型状态。
* **Best Model Saving**: Only keeps the model state that performs best on the validation set.

In [ ]:
# 定义核心的训练函数
# Define the core training function
def train_model(model, train_loader, val_loader, epochs=100, device="cpu"):
    """
    执行完整的模型训练和验证流程。
    Executes the complete model training and validation pipeline.
    """
    # --- 1. 处理类别不平衡 ---
    # --- 1. Handle Class Imbalance ---
    # 从训练数据加载器中获取所有标签
    # Get all labels from the training data loader's dataset
    train_labels = train_loader.dataset.labels
    # 计算每个类别的样本数量
    # Calculate the number of samples for each class
    class_counts = np.bincount(train_labels, minlength=len(train_loader.dataset.classes))
    # 计算每个类别的权重（样本数越少，权重越大）
    # Calculate the weight for each class (the fewer the samples, the higher the weight)
    class_weights = 1. / torch.tensor(class_counts, dtype=torch.float)
    # 对权重进行归一化处理
    # Normalize the weights
    class_weights = class_weights / class_weights.sum() * len(train_loader.dataset.classes)
    # 打印计算出的类别权重
    # Print the calculated class weights
    print(f"计算出的类别权重: {class_weights.tolist()}")

    # --- 2. 初始化损失函数、优化器和调度器 ---
    # --- 2. Initialize Loss Function, Optimizer, and Scheduler ---
    # 定义交叉熵损失函数，并传入计算好的类别权重
    # Define the CrossEntropyLoss function and pass in the calculated class weights
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
    # 定义 AdamW 优化器，传入模型参数和学习率
    # Define the AdamW optimizer, passing in the model parameters and learning rate
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
    # 定义学习率调度器，当验证损失不再下降时，它会自动降低学习率
    # Define the learning rate scheduler; it will automatically reduce the learning rate when the validation loss plateaus
    scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.1, patience=5)

    # --- 3. 初始化记录和早停变量 ---
    # --- 3. Initialize History Tracking and Early Stopping Variables ---
    # 创建一个字典来存储训练和验证过程中的损失和准确率
    # Create a dictionary to store the loss and accuracy history during training and validation
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    # 初始化最佳验证准确率为0.0
    # Initialize the best validation accuracy to 0.0
    best_val_acc = 0.0
    # 初始化早停计数器为0
    # Initialize the early stopping counter to 0
    patience_counter = 0
    # 设置早停的耐心值，即连续15轮验证准确率没有提升就停止训练
    # Set the patience for early stopping; training will stop if validation accuracy does not improve for 15 consecutive epochs
    patience = 15 
    # 初始化最佳模型所在的轮次为0
    # Initialize the epoch number of the best model to 0
    best_epoch = 0

    # --- 4. 主训练循环 ---
    # --- 4. Main Training Loop ---
    for epoch in range(epochs): # 对设定的总轮数进行循环
        # Loop over the specified total number of epochs
        model.train() # 将模型设置为训练模式
        # Set the model to training mode
        train_loss, train_acc = 0.0, 0.0 # 初始化当前轮的训练损失和准确率
        # Initialize the training loss and accuracy for the current epoch

        # 使用 tqdm 创建一个进度条，用于可视化训练过程
        # Use tqdm to create a progress bar for visualizing the training process
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Training]")
        for data, labels in pbar: # 遍历训练数据加载器中的所有批次
            # Iterate over all batches in the training data loader
            data, labels = data.to(device).float(), labels.to(device).long() # 将数据和标签移动到设备，并设置正确的数据类型
            # Move the data and labels to the device and set the correct data types
            
            # 前向传播 -> 计算损失 -> 反向传播 -> 更新参数
            # Forward pass -> Calculate loss -> Backward pass -> Update parameters
            outputs = model(data) # 1. 前向传播：将数据输入模型得到输出
            # 1. Forward Pass: Feed the data into the model to get outputs
            loss = criterion(outputs, labels) # 2. 计算损失：比较模型输出和真实标签
            # 2. Calculate Loss: Compare the model's outputs with the true labels
            optimizer.zero_grad() # 在反向传播前，清空之前的梯度
            # Before the backward pass, clear any existing gradients
            loss.backward() # 3. 反向传播：计算损失相对于模型参数的梯度
            # 3. Backward Pass: Calculate the gradients of the loss with respect to the model parameters
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) # 梯度裁剪：防止梯度爆炸，稳定训练
            # Gradient Clipping: Prevent exploding gradients to stabilize training
            optimizer.step() # 4. 更新参数：优化器根据梯度更新模型的权重
            # 4. Update Parameters: The optimizer updates the model's weights based on the gradients

            train_loss += loss.item() # 累加当前批次的损失值
            # Accumulate the loss value of the current batch
            train_acc += (outputs.argmax(1) == labels).float().mean().item() # 计算并累加当前批次的准确率
            # Calculate and accumulate the accuracy of the current batch
            pbar.set_postfix(loss=loss.item()) # 在进度条后面显示当前的损失值
            # Display the current loss value next to the progress bar

        # --- 5. 验证循环 ---
        # --- 5. Validation Loop ---
        model.eval() # 将模型设置为评估模式
        # Set the model to evaluation mode
        val_loss, val_acc = 0.0, 0.0 # 初始化当前轮的验证损失和准确率
        # Initialize the validation loss and accuracy for the current epoch
        with torch.no_grad(): # 在此块内不计算梯度，以节省资源
            # Disable gradient calculation within this block to save resources
            for data, labels in val_loader: # 遍历验证数据加载器
                # Iterate over the validation data loader
                data, labels = data.to(device).float(), labels.to(device).long() # 将数据和标签移动到设备
                # Move the data and labels to the device
                outputs = model(data) # 前向传播
                # Forward pass
                loss = criterion(outputs, labels) # 计算损失
                # Calculate the loss
                val_loss += loss.item() # 累加验证损失
                # Accumulate the validation loss
                val_acc += (outputs.argmax(1) == labels).float().mean().item() # 累加验证准确率
                # Accumulate the validation accuracy
        
        # --- 6. 计算并记录平均指标 ---
        # --- 6. Calculate and Record Average Metrics ---
        avg_train_loss = train_loss / len(train_loader) # 计算平均训练损失
        # Calculate the average training loss
        avg_train_acc = train_acc / len(train_loader) # 计算平均训练准确率
        # Calculate the average training accuracy
        avg_val_loss = val_loss / len(val_loader) # 计算平均验证损失
        # Calculate the average validation loss
        avg_val_acc = val_acc / len(val_loader) # 计算平均验证准确率
        # Calculate the average validation accuracy

        history['train_loss'].append(avg_train_loss) # 记录当前轮的平均训练损失
        # Record the average training loss for the current epoch
        history['train_acc'].append(avg_train_acc) # 记录当前轮的平均训练准确率
        # Record the average training accuracy for the current epoch
        history['val_loss'].append(avg_val_loss) # 记录当前轮的平均验证损失
        # Record the average validation loss for the current epoch
        history['val_acc'].append(avg_val_acc) # 记录当前轮的平均验证准确率
        # Record the average validation accuracy for the current epoch

        # 打印当前轮的训练和验证结果总结
        # Print a summary of the training and validation results for the current epoch
        print(f"\nEpoch {epoch+1} Summary: Train Loss: {avg_train_loss:.4f}, Acc: {avg_train_acc:.4f} | Val Loss: {avg_val_loss:.4f}, Acc: {avg_val_acc:.4f}")
        
        # 调度器根据验证损失调整学习率
        # The scheduler adjusts the learning rate based on the validation loss
        scheduler.step(avg_val_loss)

        # --- 7. 早停与最佳模型保存 ---
        # --- 7. Early Stopping and Best Model Saving ---
        if avg_val_acc > best_val_acc: # 如果当前验证准确率高于历史最佳
            # If the current validation accuracy is higher than the best so far
            best_val_acc = avg_val_acc # 更新最佳验证准确率
            # Update the best validation accuracy
            best_epoch = epoch + 1 # 更新最佳轮次
            # Update the best epoch number
            torch.save(model.state_dict(), 'best_model.pth') # 保存当前模型的权重
            # Save the current model's state dictionary (weights)
            print(f"--> 新的最佳模型已保存，验证准确率: {best_val_acc:.4f}") # 打印保存信息
            # Print a message indicating a new best model has been saved
            patience_counter = 0 # 重置早停计数器
            # Reset the early stopping counter
        else:
            patience_counter += 1 # 如果没有提升，则增加早停计数器
            # If there's no improvement, increment the early stopping counter
            if patience_counter >= patience: # 如果计数器达到耐心值
                # If the counter reaches the patience limit
                print(f"早停触发！连续 {patience} 轮验证准确率没有提升。") # 打印早停信息
                # Print the early stopping trigger message
                break # 退出训练循环
                # Break the training loop

    # --- 8. 训练结束，可视化与总结 ---
    # --- 8. End of Training, Visualization, and Summary ---
    plot_curves(history['train_loss'], history['val_loss'], history['train_acc'], history['val_acc']) # 绘制训练曲线
    # Plot the training curves
    print("\n" + "="*50) # 打印分隔线
    # Print a separator line
    print("训练结束总结") # 打印总结标题
    # Print the summary title
    print("="*50) # 打印分隔线
    # Print a separator line
    if best_epoch > 0: # 检查是否有模型被保存过
        # Check if any model was saved
        print(f"最佳模型来自第 {best_epoch} 轮") # 打印最佳模型所在的轮次
        # Print the epoch number of the best model
        print(f"最高验证准确率: {best_val_acc:.4f}") # 打印最高验证准确率
        # Print the highest validation accuracy
        print("模型权重已保存至: 'best_model.pth'") # 打印模型保存路径
        # Print the path where the model was saved
        plot_confusion_matrix(model, val_loader, device, train_loader.dataset.classes) # 绘制最终的混淆矩阵
        # Plot the final confusion matrix
    else:
        # 如果没有模型被保存，打印提示信息
        # If no model was saved, print a message
        print("在训练过程中，验证准确率并未提升，未保存任何模型。")
    print("="*50 + "\n") # 打印分隔线
    # Print a separator line

# 打印一条消息，确认函数已成功定义
# Print a message to confirm the function has been successfully defined
print("核心训练函数 'train_model' 定义成功！")

核心训练函数 'train_model' 定义成功！


## 步骤五：整合所有模块，开始训练！

## Step 5: Integrate All Modules and Start Training!

万事俱备！

Everything is ready!

现在我们来到了最激动人心的时刻。

Now we come to the most exciting moment.

我们将编写一个 `main` 函数，它将负责：

We will write a `main` function that will be responsible for:

1  设置运行设备（GPU或CPU）。

1.  Setting up the execution device (GPU or CPU).

2  创建**训练**和**验证**数据集的实例和数据加载器。

2.  Creating instances and data loaders for the **training** and **validation** datasets.

3  实例化我们的 `EEGBiFormer` 模型。

3.  Instantiating our `EEGBiFormer` model.

4  调用我们刚刚编写的 `train_model` 函数，启动整个训练流程！

4.  Calling the `train_model` function we just wrote to kick off the entire training process!

**请注意**：训练过程可能需要一些时间，具体取决于您的计算机性能，尤其是GPU性能。

**Please note**: The training process may take some time, depending on your computer's performance, especially its GPU.

In [ ]:
# 定义主函数
def main():
    # 检查 CUDA 是否可用，如果可用则使用 GPU，否则使用 CPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # 打印正在使用的设备
    print(f"Using device: {device}")
    
    # 1. 加载全部数据 (从 dataset/train)
    print("Loading all data from ./dataset/train ...")
    # 这一步会读取所有 CSV 文件
    full_dataset = EEGDataset(root_folder="./dataset/train")

    # 检查总数据量
    if len(full_dataset) == 0:
        print("Dataset is empty. Please check folder paths.")
        return

    # 2. 自动切分验证集 (80% 训练, 20% 验证)
    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size

    # 使用 random_split 切分
    # generator=torch.Generator().manual_seed(42) 确保每次切分结果一致
    train_dataset, val_dataset = torch.utils.data.random_split(
        full_dataset, 
        [train_size, val_size],
        generator=torch.Generator().manual_seed(42) 
    )

    # =============== 关键修正开始 ===============
    # Subset 对象默认没有 labels 和 data 属性，我们需要手动加上
    # 这样后续的代码（打印日志、计算权重）才能正常运行
    
    # 1. 挂载 Labels (用于计算 Class Weights)
    train_dataset.labels = full_dataset.labels[train_dataset.indices]
    val_dataset.labels = full_dataset.labels[val_dataset.indices]
    
    # 2. 挂载 Classes 和 Encoder (用于显示类别名称)
    train_dataset.classes = full_dataset.classes
    val_dataset.classes = full_dataset.classes
    train_dataset.label_encoder = full_dataset.label_encoder
    
    # 3. 挂载 Data (用于 Data Summary 打印日志 - 修复报错的核心)
    train_dataset.data = full_dataset.data[train_dataset.indices]
    val_dataset.data = full_dataset.data[val_dataset.indices]
    # =============== 关键修正结束 ===============
    
    print(f"Data split completed: {len(train_dataset)} training samples, {len(val_dataset)} validation samples.")

    # 3. 打印详细日志 (现在不会报错了，因为我们已经手动加上了 .data)
    print("===================== Data Summary =====================")
    
    # 创建类别映射字典
    class_mapping = dict(zip(full_dataset.label_encoder.classes_, full_dataset.label_encoder.transform(full_dataset.label_encoder.classes_)))
    print(f"Classes found and encoded: {class_mapping}")
    
    print("\n[Train Set]")
    print(f"  - Samples:    {len(train_dataset)}")
    print(f"  - Shape:      {train_dataset.data.shape}") # 之前报错的地方
    print(f"  - Labels:     {np.bincount(train_dataset.labels)}")
    print(f"  - Data Range: Min={train_dataset.data.min():.4f}, Max={train_dataset.data.max():.4f}")

    print("\n[Validation Set]")
    print(f"  - Samples:    {len(val_dataset)}")
    print(f"  - Shape:      {val_dataset.data.shape}")
    print(f"  - Labels:     {np.bincount(val_dataset.labels)}")
    print(f"  - Data Range: Min={val_dataset.data.min():.4f}, Max={val_dataset.data.max():.4f}")
    print("========================================================")

    # 4. 创建DataLoader
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True) 
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
    
    # 获取类别的总数
    num_classes = len(full_dataset.classes)
    
    # 实例化模型
    model = EEGBiFormer(num_classes=num_classes).to(device)
    
    # 开始训练
    train_model(model, train_loader, val_loader, device=device)

# 程序的入口点
# 检查当前脚本是否是作为主程序直接运行
if __name__ == "__main__":
    # 如果是，则调用 main() 函数
    main()

## 最终总结与展望
## Final Summary and Outlook

祝贺你完成了整个EEG项目数据处理、特征提取、模型搭建和训练评估的核心设计流程的学习！

Congratulations on completing the core workflow of the entire EEG classification project!

通过这三个模块的学习，你已经掌握了从零开始构建一个深度学习项目的全过程：

Through these three modules, you have mastered the entire process of building a deep learning project from scratch:

- **模块一**：我们学会了如何对EEG时间序列数据进行**预处理**，并构建了高效的**数据加载器**。
- **Module 1**: We learned how to **preprocess** EEG time-series data and build an efficient **data loader**.
- **模块二**：我们深入理解并亲手实现了一个**CNN+Transformer混合模型**，专门用于捕捉EEG信号的特征。
- **Module 2**: We gained a deep understanding of and implemented a **CNN+Transformer hybrid model** specifically designed to capture the features of EEG signals.
- **模块三**：我们编写了一个包含多种**高级技巧**的**训练循环**，成功训练了我们的模型，并学会了如何通过**可视化图表**来科学地评估模型性能。
- **Module 3**: We wrote a **training loop** incorporating several **advanced techniques**, successfully trained our model, and learned how to scientifically evaluate its performance through **visual charts**.
